[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Aggregating


## What you will be able to do

Ask a GroupBy for several summaries at once, with column names you choose, write a summary of your own,
and put each group's value back beside every row with `transform`.

## The idea

### The problem

The **groupby** notebook asked for one summary at a time. A report on the stations needs several,
such as each station's lowest reading, its mean, its highest, how many hours it covers and how many
it reported. Five separate calls give five Series, all named `celsius`, which then have to be put side
by side and given names by hand, as the **groupby** notebook did for `size` and `count`.

A second kind of question needs each group's value beside every row. A reading of 8 degrees is close
to Svalbard's warmest hour of the year and ordinary for Bergen, so a fair comparison measures each reading from its own station's
mean. That takes a column of 35,040 means, one for every reading, and a plain `mean()` after `groupby`
returns four.

### What aggregating and transforming are

> **Aggregating** reduces each group to one value per summary, so the result has a row for every group.
> **`agg`** asks for several summaries at once, given as a list, as a dictionary of columns, or as
> named pairs of a column and a summary. **`transform`** works out a summary for each group and returns
> it repeated for every row of that group, so its result has the same index as the table.

### Why it works that way

A summary named as text, such as `"mean"`, runs pandas' own compiled version. A function of your own
receives one group's values as a Series and must return a single value, and in a list `agg` names its
column after the function.

Named aggregation is the form that gives a report its column names. `lowest=("celsius", "min")` says:
make a column called `lowest` from the `min` of `celsius`. A dictionary such as
`{"celsius": ["min", "max"]}` asks for the same summaries, and labels the columns with two levels, the
column and the summary.

`transform` keeps the table's index, which is what makes its result usable as a column. The **Adding
and Changing Columns** notebook put a plain mean into a column and got 35,040 `NaN`, because a Series
indexed by station does not line up with rows numbered from 0. The same mean from `transform` is
indexed by row, so it subtracts from the readings directly.

`filter` is the third tool. It runs a test on each group and keeps or drops the group's rows whole.

### Where this shows up

A report with several statistics per group is the most common output of an analysis, and SQL writes
it as several aggregate functions in one `SELECT`. Measuring each value from a mean and dividing by a
standard deviation, as the standardizing example below does within each station, is what the scaling
in the **scikit-learn, Deep Dive** guide will do to a whole column. The **Reshaping** notebook's `pivot_table` takes the same summaries as `agg`.

### What this notebook covers

- `agg` with a list, with a dictionary and with named pairs
- a summary function of your own
- two steps of aggregation, days first and then stations
- `transform`, and each reading's distance from its station's mean
- `filter`, which keeps or drops whole groups
- five errors, one of them silent

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import pandas as pd

readings = pd.DataFrame({
    "station": ["Oslo", "Bergen", "Oslo", "Bergen"],
    "celsius": [6.1, 8.2, 5.8, 7.9],
})
grouped = readings.groupby("station")["celsius"]
print(grouped.agg(["min", "max"]))
print(grouped.transform("mean").round(2).tolist())
```

```
         min  max
station
Bergen   7.9  8.2
Oslo     5.8  6.1
[5.95, 8.05, 5.95, 8.05]
```

`agg` gave two summaries for each of the two stations, one row per group. `transform("mean")` gave four
values, one for every row of the table, and every value is the mean of its row's station.

## Setup

One library, three helpers for building the readings, and two tables.

- `pandas`, imported as `pd`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings

`readings` holds every hour of 2025 at the four stations, by the same formula as the **sqlite3, Deep
Dive** guide, with Svalbard's 24 silent hours on 2 March as `NaN`. `stations` holds each station's
latitude, and Kirkenes, which has no readings.

In [1]:
import math
from datetime import datetime, timedelta

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


## Worked examples

### Several summaries with a list

In [2]:
grouped = readings.groupby("station")["celsius"]

grouped.agg(["min", "mean", "max", "count"]).round(2)

,min,mean,max,count
station,,,,
Bergen,-4.8,8.00,20.8,8760
Oslo,-6.3,6.50,19.3,8760
Svalbard,-17.3,-4.48,8.3,8736
Tromso,-9.2,3.50,16.2,8760


One row for each station and one column for each summary, named after it. Every station's mean is close
to the middle of its range, as the formula makes it, and Svalbard's count is 24 short.

### A dictionary of columns

On the whole GroupBy, a dictionary says which summaries to take of which column:

In [3]:
span = readings.groupby("station").agg({"celsius": ["min", "max"], "time": ["min", "max"]})

print(span.columns.tolist())
print(span[("time", "max")].iloc[0])
span

[('celsius', 'min'), ('celsius', 'max'), ('time', 'min'), ('time', 'max')]
2025-12-31 23:00:00


celsius             time                    
             min   max        min                 max
station                                              
Bergen      -4.8  20.8 2025-01-01 2025-12-31 23:00:00
Oslo        -6.3  19.3 2025-01-01 2025-12-31 23:00:00
Svalbard   -17.3   8.3 2025-01-01 2025-12-31 23:00:00
Tromso      -9.2  16.2 2025-01-01 2025-12-31 23:00:00

The columns have two levels, the column and the summary, and a pair such as `("time", "max")` picks
one of them, which the second printed line read for Bergen. A pair is awkward to type, and the next
form avoids it.

### Named aggregation

Each keyword names a column of the result, and its value is a pair of the column to read and the
summary to take:

In [4]:
report = readings.groupby("station").agg(
    lowest=("celsius", "min"),
    mean=("celsius", "mean"),
    highest=("celsius", "max"),
    hours=("celsius", "size"),
    reported=("celsius", "count"),
)
report.round(2)

,lowest,mean,highest,hours,reported
station,,,,,
Bergen,-4.8,8.00,20.8,8760,8760
Oslo,-6.3,6.50,19.3,8760,8760
Svalbard,-17.3,-4.48,8.3,8760,8736
Tromso,-9.2,3.50,16.2,8760,8760


The report has the columns it was given, in the order they were written. `hours` and `reported` differ
only for Svalbard, by the 24 hours it was silent.

### A summary of your own

A function that takes a Series and returns one value can be a summary. In a list, `agg` names its column
after the function:

In [5]:
def spread(values):
    """The difference between the highest and the lowest value."""
    return values.max() - values.min()


print(grouped.agg(["mean", spread, lambda values: values.max()]).columns.tolist())
grouped.agg(spread).round(1)

['mean', 'spread', '<lambda_0>']


station
Bergen      25.6
Oslo        25.6
Svalbard    25.6
Tromso      25.4
Name: celsius, dtype: float64

A function written with `def` keeps its name as the column's, and every `lambda` is named `<lambda>`,
so in a list with other functions pandas numbers them, `<lambda_0>` here. Named aggregation, as in `range=("celsius", spread)`, sets the name either way.
Bergen's, Oslo's and Svalbard's years span 25.6 degrees, and Tromso's 25.4.

### A daily range for each station

Some questions need two steps. The mean daily range, the gap between a day's lowest and highest reading
averaged over the year, needs a range for every station and day, and then a mean of those for every
station:

In [6]:
daily = readings.groupby(["station", readings["time"].dt.date])["celsius"].agg(["min", "max"])
daily["range"] = (daily["max"] - daily["min"]).round(1)

print(len(daily), "rows, one for every station and day")
print(daily.head(3))
print(daily["range"].isna().sum(), "day with no range")
daily.groupby("station")["range"].mean().round(2)

1460 rows, one for every station and day
                    min  max  range
station time                       
Bergen  2025-01-01 -3.7  2.8    6.5
        2025-01-02 -3.9  3.0    6.9
        2025-01-03 -4.5  3.0    7.5
1 day with no range


station
Bergen      6.90
Oslo        6.90
Svalbard    6.90
Tromso      6.89
Name: range, dtype: float64

`daily` has a row for every station and day, 1,460 in all. Svalbard's 2 March has no readings, so its
lowest, highest and range are `NaN`, and the second `mean` left that day out. Every station's days swing
by about 6.9 degrees, which is the formula's daily swing of 3 degrees either side of the mean, plus its
small wobble.

### transform keeps the table's shape

In [7]:
station_mean = grouped.transform("mean")

print(len(grouped.mean()), "values from mean,", len(station_mean), "from transform")

table = readings.assign(anomaly=(readings["celsius"] - station_mean).round(2))
table.head(4)

4 values from mean, 35040 from transform


,station,time,celsius,anomaly
0,Bergen,2025-01-01,-3.6,-11.60
1,Oslo,2025-01-01,-3.5,-10.00
2,Svalbard,2025-01-01,-14.6,-10.12
3,Tromso,2025-01-01,-6.7,-10.20


`transform("mean")` returned 35,040 values, each the mean of its row's station, indexed like `readings`,
so subtracting it from `celsius` lined each mean up with its own row. `anomaly` is how far each reading is from its own
station's mean. Here is each station's largest anomaly:

In [8]:
table.loc[table.groupby("station")["anomaly"].idxmax()]

,station,time,celsius,anomaly
19356,Bergen,2025-07-21 15:00:00,20.8,12.80
18973,Oslo,2025-07-17 15:00:00,19.3,12.80
18590,Svalbard,2025-07-13 15:00:00,8.3,12.78
18207,Tromso,2025-07-09 15:00:00,16.2,12.70


Svalbard's warmest hour was 8.3 degrees and Bergen's 20.8, and both are about 12.8 degrees above their
own stations' means. The formula gives every station the same seasonal and daily swings around a
different mean, and the anomaly is what shows it.

### transform with a function

A function given to `transform` receives each group's values and returns one value per row. Here each
reading is measured from its station's mean in units of its station's standard deviation:

In [9]:
standardized = grouped.transform(lambda values: (values - values.mean()) / values.std())

print(standardized.head(4).round(2).tolist())
print(standardized.groupby(readings["station"]).std().round(2).tolist())

[-1.72, -1.49, -1.5, -1.52]
[1.0, 1.0, 1.0, 1.0]


After the change, every station's readings have a standard deviation of 1, so -1.5 means a reading one
and a half standard deviations below its own station's mean, at Svalbard as at Bergen.

### filter keeps or drops whole groups

`filter` runs a test on each group's rows and keeps the groups where the test is true:

In [10]:
mild = readings.groupby("station").filter(lambda group: group["celsius"].mean() > 0)

print(len(mild), "rows from", mild["station"].unique().tolist())

26280 rows from ['Bergen', 'Oslo', 'Tromso']


Svalbard's mean is below zero, so all 8,760 of its rows were dropped, and the other three stations kept
all of theirs.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/13-aggregating-solutions.ipynb).

**1.** With a list in `agg`, print each station's median and standard deviation of `celsius`, rounded to two places.

In [11]:
# your code here

**2.** With named aggregation, make a table of each station's `first_hour` and `last_hour`, the earliest and latest values of `time`.

In [12]:
# your code here

**3.** Write a function `frost_hours` that counts the values below zero in a Series, and use it in `agg` for every station.

In [13]:
# your code here

**4.** Add a column `monthly_mean` to a copy of `readings`, holding each station's mean for the month of each reading, with `transform` on a groupby of station and month, and show the first three rows.

In [14]:
# your code here

**5.** Use `filter` to keep the stations whose lowest reading is below -8, and print which stations remain.

In [15]:
# your code here

## Common errors

### SpecificationError: nested renamer is not supported

In [16]:
readings.groupby("station").agg({"celsius": {"lowest": "min"}})

SpecificationError: nested renamer is not supported

pandas refuses a dictionary inside the dictionary, which was meant to name the result. Named
aggregation does that job:

In [17]:
readings.groupby("station").agg(lowest=("celsius", "min"))

,lowest
station,
Bergen,-4.8
Oslo,-6.3
Svalbard,-17.3
Tromso,-9.2


### KeyError: "Column(s) ['Celsius'] do not exist"

In [18]:
readings.groupby("station").agg(lowest=("Celsius", "min"))

KeyError: "Column(s) ['Celsius'] do not exist"

The first item of each pair is a column name, matched exactly, and the column is `celsius`. The message
lists every name it could not find.

### ValueError: Must produce aggregated value

In [19]:
grouped.agg(lambda values: values.head(2))

ValueError: Must produce aggregated value

A summary given to `agg` must reduce a group to one value, and `head(2)` returns two. Use `transform`
for one value per row, or `apply` for anything else, as the **groupby** notebook did.

### No error, and NaN in all but two rows of each group

In [20]:
firsts = grouped.transform(lambda values: values.head(2))

print(firsts.notna().sum(), "of", len(firsts), "values are not NaN")

8 of 35040 values are not NaN


`transform` did not raise. It lined the two values each group returned up with the group's rows by
label and filled the other 8,758 rows of every station with `NaN`. A function given to `transform`
should return one value per row, as the standardizing example did, or a single value, which
`transform` repeats.

### TypeError: filter function returned a Series, but expected a scalar bool

In [21]:
readings.groupby("station").filter(lambda group: group["celsius"] > 0)

TypeError: filter function returned a Series, but expected a scalar bool

`filter` keeps or drops a whole group, so its test must give one `True` or `False` for the group, and
`group["celsius"] > 0` gives one for every row. Reduce the test to one value, as with
`(group["celsius"] > 0).all()` or `group["celsius"].mean() > 0`. To keep chosen rows inside every
group, use a mask, which the **Boolean Filtering** notebook covers.

## Recap

- Ask for several summaries at once with `agg` and a list, a dictionary, or named pairs
- Name a report's columns with named aggregation, as in `lowest=("celsius", "min")`
- Write a summary function that takes a Series and returns one value
- Put each group's value beside every row with `transform`, and subtract it to measure each reading
  from its group
- Keep or drop whole groups with `filter` and a test that returns one value

## What is next

The **Reshaping** notebook turns the long readings into a wide table of stations by month, and back,
and shows which questions each shape makes easier.

---

&#8592; **Previous:** [groupby](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/12-groupby.ipynb)  &nbsp;·&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Reshaping](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/14-reshaping.ipynb) &#8594;
